<a href="https://colab.research.google.com/github/Angelinat3/Machine-Learning2026/blob/main/JS06/JS06_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import cv2
import glob
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Load dataset
df = pd.read_csv('voice.csv')

# Separate features and target
X = df.drop(columns=['label'])
y = df['label']

# 2. Feature Scaling (Crucial for SVM performance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Model Training & Evaluation
splits = [(0.30, '70:30'), (0.20, '80:20')]
kernels = ['linear', 'poly', 'rbf']

results = []

for test_size, split_label in splits:
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42, stratify=y
    )
    for kernel in kernels:
        model = SVC(kernel=kernel)
        model.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))

        results.append({
            'Split': split_label,
            'Kernel': kernel.capitalize(),
            'Train Accuracy': f"{train_acc * 100:.2f}%",
            'Test Accuracy': f"{test_acc * 100:.2f}%"
        })

# Display Results
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Split Kernel Train Accuracy Test Accuracy
70:30 Linear         97.20%        97.90%
70:30   Poly         96.80%        95.90%
70:30    Rbf         98.38%        98.32%
80:20 Linear         97.63%        97.48%
80:20   Poly         96.84%        95.74%
80:20    Rbf         98.42%        98.26%


In [7]:
import zipfile

# Ekstrak file images.zip
with zipfile.ZipFile('images.zip', 'r') as zip_ref:
    zip_ref.extractall('.')
print("Ekstrak selesai!")

Ekstrak selesai!


In [11]:
# 1. Feature Extraction: HSV Color Histogram
def extract_hsv_histogram(image_path, bins=(8, 8, 8)):
    image = cv2.imread(image_path)
    if image is None:
        return None

    # Resize image for standardization
    image = cv2.resize(image, (600, 400))

    # Convert to HSV color space
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Compute 3D HSV Histogram
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])

    # Normalize histogram values
    cv2.normalize(hist, hist)
    return hist.flatten()

# 2. Load Dataset
def load_data(data_dir):
    features = []
    labels = []

    # Label mapping: day = 1, night = 0
    categories = {'day': 1, 'night': 0}

    for category, label in categories.items():
        folder_path = os.path.join(data_dir, category)

        # Ensure the directory exists before listing
        if not os.path.exists(folder_path):
            continue

        for img_name in os.listdir(folder_path):
            # Ignore hidden macOS metadata files (starting with '._')
            if img_name.startswith('._'):
                continue

            img_path = os.path.join(folder_path, img_name)
            if img_path.lower().endswith(('.jpg', '.png', '.jpeg')):
                hist_feat = extract_hsv_histogram(img_path)
                if hist_feat is not None:
                    features.append(hist_feat)
                    labels.append(label)

    return np.array(features), np.array(labels)

# Set path to your extracted dataset folder
DATASET_PATH = '/content/images/test'  # Contains 'day' and 'night' subfolders

X, y = load_data(DATASET_PATH)

# Check if data was loaded correctly
if len(X) == 0:
    raise ValueError("No images were loaded! Check your directory structure.")

# 3. Train-Test Split (80:20 Ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Hyperparameter Tuning for RBF Kernel (C and gamma)
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

# Best Model
best_model = grid_search.best_estimator_

# 5. Evaluation
y_train_pred = best_model.predict(X_train_scaled)
y_test_pred = best_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Best Hyperparameters : {grid_search.best_params_}")
print(f"Train Accuracy       : {train_acc * 100:.2f}%")
print(f"Test Accuracy        : {test_acc * 100:.2f}%")
print("\nClassification Report (Test Data):")
print(classification_report(y_test, y_test_pred, target_names=['Night (0)', 'Day (1)']))

Best Hyperparameters : {'C': 10, 'gamma': 'auto'}
Train Accuracy       : 100.00%
Test Accuracy        : 78.12%

Classification Report (Test Data):
              precision    recall  f1-score   support

   Night (0)       0.70      1.00      0.82        16
     Day (1)       1.00      0.56      0.72        16

    accuracy                           0.78        32
   macro avg       0.85      0.78      0.77        32
weighted avg       0.85      0.78      0.77        32

